In [33]:
# Import required libraries
import pandas as pd
import dash
from dash import html
from dash import dcc
from dash.dependencies import Input, Output
import plotly.express as px

# Read the airline data into pandas dataframe
spacex_df = pd.read_csv("spacex_launch_dash.csv")
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

# Create a dash application
app = dash.Dash(__name__)

# Create an app layout
app.layout = html.Div(children=[html.H1('SpaceX Launch Records Dashboard',
                                        style={'textAlign': 'center', 'color': '#503D36',
                                               'font-size': 40}),
                                # TASK 1: Add a dropdown list to enable Launch Site selection
                                # The default select value is for ALL sites
                                dcc.Dropdown(
                                    id='site-dropdown',
                                    options=[
                                        {'label': 'All Sites', 'value': 'ALL'},
                                        {'label': 'CCAFS LC-40', 'value': 'CCAFS LC-40'},
                                        {'label': 'KSC LC-39A', 'value': 'KSC LC-39A'},
                                        {'label': 'VAFB SLC-4E', 'value': 'VAFB SLC-4E'},
                                        {'label': 'CCAFS SLC-40', 'value': 'CCAFS SLC-40'}
                                    ],
                                    value='ALL',
                                    placeholder="Select a Launch Site here",
                                    searchable=True
                                ),
                                html.Br(),

                                # TASK 2: Add a pie chart to show the total successful launches count for all sites
                                # If a specific launch site was selected, show the Success vs. Failed counts for the site
                                html.Div(dcc.Graph(id='success-pie-chart')),
                                html.Br(),

                                html.P("Payload range (Kg):"),
                                # TASK 3: Add a slider to select payload range
                                dcc.RangeSlider(id='payload-slider',
                                    min=0, max=10000, step=1000,
                                    marks={0: '0', 10000: '10000'},
                                    value=[min_payload, max_payload]),
                                # TASK 4: Add a scatter chart to show the correlation between payload and launch success
                                html.Div(dcc.Graph(id='success-payload-scatter-chart')),
                                ])

# TASK 2:
# Add a callback function for `site-dropdown` as input, `success-pie-chart` as output

@app.callback(
    Output(component_id='success-pie-chart', component_property='figure'),
    Input(component_id='site-dropdown', component_property='value')
)
def get_pie_chart(selected_site):
    if selected_site == 'ALL':
        # Pie chart showing total successful launches by site
        fig = px.pie(spacex_df, 
                     names='Launch Site', 
                     values='class',  # assuming 'class' is 1=success, 0=failure
                     title='Total Successful Launches by Site')
    else:
        # Filter dataframe for the selected site
        filtered_df = spacex_df[spacex_df['Launch Site'] == selected_site]
        # Pie chart showing success vs failed counts
        fig = px.pie(filtered_df, 
                     names='class', 
                     title=f'Total Success vs Failure for site {selected_site}')
    return fig


# TASK 4:
# Add a callback function for `site-dropdown` and `payload-slider` as inputs, `success-payload-scatter-chart` as output

@app.callback(
    Output(component_id='success-payload-scatter-chart', component_property='figure'),
    Input(component_id='site-dropdown', component_property='value'),
    Input(component_id='payload-slider', component_property='value')
)
def update_scatter_chart(selected_site, payload_range):
    low, high = payload_range  # unpack the slider range
    
    # Filter dataframe by payload range first
    filtered_df = spacex_df[(spacex_df['Payload Mass (kg)'] >= low) &
                             (spacex_df['Payload Mass (kg)'] <= high)]
    
    if selected_site == 'ALL':
        # Scatter plot for all sites
        fig = px.scatter(
            filtered_df,
            x='Payload Mass (kg)',
            y='class',  # success=1, failure=0
            color='Booster Version Category',  # color points by booster type
            title='Payload vs. Launch Outcome for All Sites',
            hover_data=['Launch Site']
        )
    else:
        # Filter for the selected launch site
        site_df = filtered_df[filtered_df['Launch Site'] == selected_site]
        
        fig = px.scatter(
            site_df,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version Category',
            title=f'Payload vs. Launch Outcome for site {selected_site}',
            hover_data=['Launch Site']
        )
    
    return fig
# @app.callback(
#     Output(component_id='success-payload-scatter-chart', component_property='figure'),
#     Input(component_id='site-dropdown', component_property='value'),
#     Input(component_id='payload-slider', component_property='value')
# )
# def update_scatter_chart(selected_site, payload_range):
#     low, high = payload_range  # unpack the slider range
#     # Filter dataframe by payload range
#     filtered_df = spacex_df[(spacex_df['Payload Mass (kg)'] >= low) &
#                              (spacex_df['Payload Mass (kg)'] <= high)]
    
#     # Further filter by launch site if a specific site is selected
#     if selected_site != 'ALL':
#         filtered_df = filtered_df[filtered_df['Launch Site'] == selected_site]
    
#     # Create scatter plot
#     fig = px.scatter(
#         filtered_df,
#         x='Payload Mass (kg)',
#         y='class',  # 1=success, 0=failure
#         color='Booster Version Category',  # optional, categorize boosters
#         title=f"Payload vs. Launch Outcome for site {selected_site}",
#         hover_data=['Launch Site']
#     )
    
#     return fig

# Run the app
if __name__ == '__main__':
    app.run()


In [15]:
spacex_df.head()

,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


In [9]:
spacex_df['Launch Site'].value_counts()

CCAFS LC-40     26
KSC LC-39A      13
VAFB SLC-4E     10
CCAFS SLC-40     7
Name: Launch Site, dtype: int64

In [41]:
# Import required libraries
import pandas as pd
import dash
from dash import html, dcc
from dash.dependencies import Input, Output
import plotly.express as px

# Read the data
spacex_df = pd.read_csv("spacex_launch_dash.csv")
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

# Create Dash app
app = dash.Dash(__name__)

# App layout with styling and better spacing
app.layout = html.Div(style={'font-family': 'Arial, sans-serif', 'margin': '20px'}, children=[
    
    html.H1('SpaceX Launch Records Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': '48px', 'margin-bottom': '30px'}),
    
    # Row for dropdown and slider
    html.Div(style={'display': 'flex', 'justify-content': 'space-between', 'margin-bottom': '20px'}, children=[
        html.Div(style={'flex': '0 0 48%'}, children=[
            html.Label("Select Launch Site:", style={'font-weight': 'bold', 'font-size': '18px'}),
            dcc.Dropdown(
                id='site-dropdown',
                options=[
                    {'label': 'All Sites', 'value': 'ALL'},
                    {'label': 'CCAFS LC-40', 'value': 'CCAFS LC-40'},
                    {'label': 'KSC LC-39A', 'value': 'KSC LC-39A'},
                    {'label': 'VAFB SLC-4E', 'value': 'VAFB SLC-4E'},
                    {'label': 'CCAFS SLC-40', 'value': 'CCAFS SLC-40'}
                ],
                value='ALL',
                placeholder="Select a Launch Site",
                searchable=True
            )
        ]),
        
        html.Div(style={'flex': '0 0 48%'}, children=[
            html.Label("Payload range (Kg):", style={'font-weight': 'bold', 'font-size': '18px'}),
            dcc.RangeSlider(
                id='payload-slider',
                min=0, max=10000, step=1000,
                marks={0: '0', 10000: '10000'},
                value=[min_payload, max_payload]
            )
        ])
    ]),
    
    # Row for pie chart and scatter chart
    html.Div(style={'display': 'flex', 'flex-direction': 'row', 'justify-content': 'space-between'}, children=[
        html.Div(dcc.Graph(id='success-pie-chart'), style={'flex': '0 0 48%', 'margin-right': '2%'}),
        html.Div(dcc.Graph(id='success-payload-scatter-chart'), style={'flex': '0 0 48%'})
    ])
])

# Callback for pie chart
@app.callback(
    Output('success-pie-chart', 'figure'),
    Input('site-dropdown', 'value')
)
def get_pie_chart(selected_site):
    if selected_site == 'ALL':
        # Pie chart showing total successful launches by site
        fig = px.pie(
            spacex_df, 
            names='Launch Site', 
            values='class',
            title='Total Successful Launches by Site'
        )
    else:
        # Filter for the selected launch site
        filtered_df = spacex_df[spacex_df['Launch Site'] == selected_site].copy()
        # Map 0/1 to 'Failure'/'Success'
        filtered_df['Outcome'] = filtered_df['class'].map({0: 'Failure', 1: 'Success'})
        fig = px.pie(
            filtered_df, 
            names='Outcome',
            title=f'Total Success vs Failure for site {selected_site}'
        )
    
    # Add black outline for clarity
    fig.update_traces(marker=dict(line=dict(color='#000000', width=1)))
    return fig

# Callback for scatter chart
@app.callback(
    Output('success-payload-scatter-chart', 'figure'),
    Input('site-dropdown', 'value'),
    Input('payload-slider', 'value')
)
def update_scatter_chart(selected_site, payload_range):
    low, high = payload_range
    filtered_df = spacex_df[(spacex_df['Payload Mass (kg)'] >= low) &
                             (spacex_df['Payload Mass (kg)'] <= high)]
    
    if selected_site == 'ALL':
        fig = px.scatter(filtered_df, x='Payload Mass (kg)', y='class',
                         color='Booster Version Category',
                         title='Payload vs. Launch Outcome for All Sites',
                         hover_data=['Launch Site'])
    else:
        site_df = filtered_df[filtered_df['Launch Site'] == selected_site]
        fig = px.scatter(site_df, x='Payload Mass (kg)', y='class',
                         color='Booster Version Category',
                         title=f'Payload vs. Launch Outcome for site {selected_site}',
                         hover_data=['Launch Site'])
    fig.update_layout(title_font_size=20)
    return fig

# Run the app
if __name__ == '__main__':
    app.run_server(debug=True)

In [43]:
# Import required libraries
import pandas as pd
import dash
from dash import html, dcc
from dash.dependencies import Input, Output
import plotly.express as px

# Read the data
spacex_df = pd.read_csv("spacex_launch_dash.csv")
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

# Create Dash app
app = dash.Dash(__name__)

# App layout with styling and better spacing
app.layout = html.Div(style={'font-family': 'Arial, sans-serif', 'margin': '20px'}, children=[
    
    html.H1('SpaceX Launch Records Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': '48px', 'margin-bottom': '30px'}),
    
    # Row for dropdown and slider
    html.Div(style={'display': 'flex', 'justify-content': 'space-between', 'margin-bottom': '20px'}, children=[
        html.Div(style={'flex': '0 0 48%'}, children=[
            html.Label("Select Launch Site:", style={'font-weight': 'bold', 'font-size': '18px'}),
            dcc.Dropdown(
                id='site-dropdown',
                options=[
                    {'label': 'All Sites', 'value': 'ALL'},
                    {'label': 'CCAFS LC-40', 'value': 'CCAFS LC-40'},
                    {'label': 'KSC LC-39A', 'value': 'KSC LC-39A'},
                    {'label': 'VAFB SLC-4E', 'value': 'VAFB SLC-4E'},
                    {'label': 'CCAFS SLC-40', 'value': 'CCAFS SLC-40'}
                ],
                value='ALL',
                placeholder="Select a Launch Site",
                searchable=True
            )
        ]),
        
        html.Div(style={'flex': '0 0 48%'}, children=[
            html.Label("Payload range (Kg):", style={'font-weight': 'bold', 'font-size': '18px'}),
            dcc.RangeSlider(
                id='payload-slider',
                min=0, max=10000, step=1000,
                marks={0: '0', 10000: '10000'},
                value=[min_payload, max_payload]
            )
        ])
    ]),
    
    # Row for pie chart and scatter chart
    html.Div(style={'display': 'flex', 'flex-direction': 'row', 'justify-content': 'space-between'}, children=[
        html.Div(dcc.Graph(id='success-pie-chart'), style={'flex': '0 0 48%', 'margin-right': '2%'}),
        html.Div(dcc.Graph(id='success-payload-scatter-chart'), style={'flex': '0 0 48%'})
    ])
])

# Callback for pie chart
@app.callback(
    Output('success-pie-chart', 'figure'),
    Input('site-dropdown', 'value')
)
def get_pie_chart(selected_site):
    if selected_site == 'ALL':
        # For all sites, compute the success rate per site
        # Success rate = sum of successful launches / total launches per site
        site_stats = spacex_df.groupby('Launch Site')['class'].mean().reset_index()
        site_stats['Success Rate (%)'] = site_stats['class'] * 100  # convert to percentage
        fig = px.pie(
            site_stats,
            names='Launch Site',
            values='Success Rate (%)',
            title='Success Rate by Launch Site'
        )
    else:
        # Filter for the selected site
        filtered_df = spacex_df[spacex_df['Launch Site'] == selected_site].copy()
        # Map 0/1 to 'Failure'/'Success'
        filtered_df['Outcome'] = filtered_df['class'].map({0: 'Failure', 1: 'Success'})
        # Calculate counts for each outcome
        outcome_counts = filtered_df['Outcome'].value_counts(normalize=True).reset_index()
        outcome_counts.columns = ['Outcome', 'Rate']
        outcome_counts['Rate'] = outcome_counts['Rate'] * 100  # percentage
        fig = px.pie(
            outcome_counts,
            names='Outcome',
            values='Rate',
            title=f'Success vs Failure Rate for site {selected_site}'
        )
    
    # Add black outline for clarity
    fig.update_traces(marker=dict(line=dict(color='#000000', width=1)))
    return fig

# Callback for scatter chart
@app.callback(
    Output('success-payload-scatter-chart', 'figure'),
    Input('site-dropdown', 'value'),
    Input('payload-slider', 'value')
)
def update_scatter_chart(selected_site, payload_range):
    low, high = payload_range
    filtered_df = spacex_df[(spacex_df['Payload Mass (kg)'] >= low) &
                             (spacex_df['Payload Mass (kg)'] <= high)]
    
    if selected_site == 'ALL':
        fig = px.scatter(filtered_df, x='Payload Mass (kg)', y='class',
                         color='Booster Version Category',
                         title='Payload vs. Launch Outcome for All Sites',
                         hover_data=['Launch Site'])
    else:
        site_df = filtered_df[filtered_df['Launch Site'] == selected_site]
        fig = px.scatter(site_df, x='Payload Mass (kg)', y='class',
                         color='Booster Version Category',
                         title=f'Payload vs. Launch Outcome for site {selected_site}',
                         hover_data=['Launch Site'])
    fig.update_layout(title_font_size=20)
    return fig

# Run the app
if __name__ == '__main__':
    app.run_server(debug=True)

In [ ]:
@app.callback(
    Output('success-pie-chart', 'figure'),
    Input('site-dropdown', 'value')
)
def get_pie_chart(selected_site):
    if selected_site == 'ALL':
        # For all sites, compute the success rate per site
        # Success rate = sum of successful launches / total launches per site
        site_stats = spacex_df.groupby('Launch Site')['class'].mean().reset_index()
        site_stats['Success Rate (%)'] = site_stats['class'] * 100  # convert to percentage
        fig = px.pie(
            site_stats,
            names='Launch Site',
            values='Success Rate (%)',
            title='Success Rate by Launch Site'
        )
    else:
        # Filter for the selected site
        filtered_df = spacex_df[spacex_df['Launch Site'] == selected_site].copy()
        # Map 0/1 to 'Failure'/'Success'
        filtered_df['Outcome'] = filtered_df['class'].map({0: 'Failure', 1: 'Success'})
        # Calculate counts for each outcome
        outcome_counts = filtered_df['Outcome'].value_counts(normalize=True).reset_index()
        outcome_counts.columns = ['Outcome', 'Rate']
        outcome_counts['Rate'] = outcome_counts['Rate'] * 100  # percentage
        fig = px.pie(
            outcome_counts,
            names='Outcome',
            values='Rate',
            title=f'Success vs Failure Rate for site {selected_site}'
        )
    
    # Add black outline for clarity
    fig.update_traces(marker=dict(line=dict(color='#000000', width=1)))
    return fig